# MAGI Vision — Cloud Training Pipeline
Trains the **Celebi** MobileNetV2 plant-health classifier and exports `celebi.tflite` for deployment on the Pi 4B.

> ⚠️ **Before running anything**: Go to `Runtime → Change runtime type → Hardware accelerator → GPU (T4)`.
> The cell below will confirm and configure the GPU.

## Step 0: Configure GPU + Stabilise Memory
Run this **first**. Configures XLA JIT, GPU memory growth, clears any stale state, and verifies the runtime is GPU.

**Why this matters on free T4 Colab:**
- Free T4 has ~12 GB CPU RAM and ~12 GB VRAM
- Without `memory_growth=True`, TensorFlow pre-allocates the **full** VRAM upfront, leaving no room for CPU↔GPU tensor transfers
- The `gc.collect()` clears Python objects from any previous failed run that could be holding RAM

In [ ]:
import os, gc, subprocess

# ── Suppress verbose cuDNN/XLA C++ INFO spam before importing TF ──────────────
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'           # hide INFO + most WARNINGs
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'  # XLA JIT

import tensorflow as tf

# ── Clear any stale session from a previous crashed run ───────────────────────
tf.keras.backend.clear_session()
gc.collect()

# ── GPU memory growth: MUST be set before any TF graph is built ───────────────
# Without this, TF grabs all ~12 GB VRAM at once, leaving no headroom.
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(f'  ⚠️  memory_growth already set ({e})')

print(f'TensorFlow version : {tf.__version__}')
print(f'GPUs detected      : {gpus}')

if not gpus:
    print()
    print('❌  NO GPU DETECTED — training will be 50–100× slower on CPU!')
    print('   → Go to Runtime > Change runtime type > Hardware accelerator > GPU')
    print('   → Then re-run this cell to confirm.')
else:
    print()
    print(f'✅  GPU active: {gpus[0].name}')
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
             '--format=csv,noheader']
        ).decode()
        name, mem_total, mem_free = out.strip().split(',')
        print(f'   Model      : {name.strip()}')
        print(f'   VRAM total : {mem_total.strip()}')
        print(f'   VRAM free  : {mem_free.strip()}')
    except Exception:
        pass
    print()
    print('🚀  Optimisations active for this run (tuned for free T4 / 12 GB RAM):')
    print('   • GPU memory growth enabled    (prevents full VRAM pre-allocation)')
    print('   • XLA JIT compilation          (~15–20% throughput gain)')
    print('   • Mixed precision float16      (~2× speed, half VRAM per layer)')
    print('   • Batch size 64                (safe for 12 GB CPU RAM + shuffle buffer)')
    print('   • Shuffle buffer 1024 samples  (was 4096 — caused OOM crash)')
    print('   • Val set NOT cached in RAM    (was cache=True — caused OOM crash)')
    print('   • Sharded TFRecord I/O         (4 parallel read streams)')
    print('   • steps_per_execution=5        (lower peak memory spike per step)')
    print('   • CosineDecay LR (Phase 2)     (better fine-tune convergence)')
    print()
    print('⏱  Estimated total training time on free T4: ~25–35 min')

## Step 0b: RAM Health Check
Run this before training to confirm you have enough headroom. If free RAM < 4 GB, **restart the runtime** (`Runtime → Restart runtime`) and re-run Step 0 first.

In [ ]:
import psutil, gc

gc.collect()  # Force Python GC before measuring

vm = psutil.virtual_memory()
total_gb  = vm.total  / 1024**3
used_gb   = vm.used   / 1024**3
free_gb   = vm.available / 1024**3

print(f'CPU RAM  total : {total_gb:.1f} GB')
print(f'CPU RAM  used  : {used_gb:.1f} GB')
print(f'CPU RAM  free  : {free_gb:.1f} GB')
print()

# Minimum needed: ~1024 shuffle buffer × ~1.6 MB/sample (float32 expand) ≈ 1.6 GB
# + model weights + OS overhead → 4 GB headroom is the safe minimum
MIN_FREE_GB = 4.0
if free_gb < MIN_FREE_GB:
    print(f'⚠️  WARNING: Only {free_gb:.1f} GB free — below the {MIN_FREE_GB} GB minimum.')
    print('   → Go to Runtime → Restart runtime, then re-run Step 0 before training.')
else:
    print(f'✅  Sufficient RAM available ({free_gb:.1f} GB free ≥ {MIN_FREE_GB} GB minimum)')
    print('   Safe to proceed to training.')

## Step 1: Upload & Extract the Project
1. On your local machine, run:
   ```bash
   cd /home/aki/Downloads/MAGI/HeatMAP/MLOPs-Production-PIPELINE
   zip -r magi_project.zip . -x "venv/*" -x "artifact/*" -x ".git/*" -x "__pycache__/*"
   ```
2. Upload `magi_project.zip` via the Colab left sidebar (📁 Files tab).
3. Run the cell below.

In [ ]:
import os

if not os.path.exists('magi_project'):
    print('Extracting magi_project.zip...')
    !unzip -q magi_project.zip -d magi_project
    print('Done.')
else:
    print('magi_project/ already exists — skipping extraction.')

%cd magi_project
print(f'Working directory: {os.getcwd()}')

## Step 2: Install Dependencies

In [ ]:
print('Installing requirements...')
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q psutil   # for RAM health check cell above
print('✅ Dependencies installed.')

## Step 3: Set Kaggle API Credentials
Replace the placeholders with your [Kaggle API key](https://www.kaggle.com/settings/account).

In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = "your_kaggle_username"  # ← replace
os.environ['KAGGLE_KEY']      = "your_kaggle_api_key"   # ← replace

if os.environ['KAGGLE_USERNAME'] == 'your_kaggle_username':
    print('⚠️  Kaggle credentials not set! Edit the cell above before running the pipeline.')
else:
    print(f'✅  Kaggle credentials set for: {os.environ["KAGGLE_USERNAME"]}')

## Step 4: Run the Training Pipeline
This will:
1. Download the Cotton dataset from Kaggle
2. Validate & transform data (parallel spectral preprocessing → sharded TFRecords)
3. Train the 8-channel MobileNetV2 (Phase 1 head + Phase 2 fine-tune w/ CosineDecay)
4. Evaluate and export `celebi.tflite`

**Expected times on free Colab T4 (batch_size=64, with all OOM fixes applied):**
| Stage | Estimated Time |
|---|---|
| Data ingestion | ~5–8 min |
| Data transformation (parallel) | ~1–2 min |
| Phase 1 training (12 epochs, early stop) | ~5–8 min |
| Phase 2 fine-tuning (20 epochs, early stop) | ~10–16 min |
| TFLite conversion | ~1 min |
| **Total** | **~25–35 min** |

> 💡 **If the runtime crashes mid-training**: Go to `Runtime → Restart runtime`, re-run Step 0 (GPU setup + RAM check), then re-run this cell. The pipeline will re-download data. Consider upgrading to **Colab Pro** to get priority GPU allocation and more RAM.

In [ ]:
import gc, tensorflow as tf

# Final GPU sanity check before the long run
assert tf.config.list_physical_devices('GPU'), (
    'NO GPU! Switch runtime to GPU (Runtime > Change runtime type) before running.'
)
print(f'✅ GPU confirmed — launching pipeline on {tf.config.list_physical_devices("GPU")[0].name}')

# Clean up any Python objects from imports / previous runs before heavy training
gc.collect()
print()

from magi_vision.pipeline.training_pipeline import MAGITrainPipeline

pipeline = MAGITrainPipeline()
pipeline.run_pipeline()

## Step 5: Download the Trained Model
Downloads `celebi.tflite` **and** `normalization_stats.json` to your local machine.

Copy both to your Pi deployment directory (`/opt/magi/models/`).

In [ ]:
from google.colab import files
import os

# Both files are required for the Celebi node on the Pi
downloads = [
    'tflite_export/celebi.tflite',
    'artifact/data_transformation/transform_config/normalization_stats.json',
]

for path in downloads:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'📦 Downloading {os.path.basename(path)} ({size_kb:.0f} KB)...')
        files.download(path)
    else:
        print(f'❌ Not found: {path}  (did training complete successfully?)')